# QuickPay Fintech Pipeline — Parts 3 & 4
**Part 3:** Python Reconciliation Workflow (ledger.csv vs gateway.csv)  
**Part 4:** JSON Normalization (api_response_sample.json)

---

In [1]:
import pandas as pd
import numpy as np
import json, os

RAW  = os.path.join('..', '01_data', 'raw')
PROC = os.path.join('..', '01_data', 'processed')
os.makedirs(PROC, exist_ok=True)
print('Libraries loaded. Ready to begin.')

Libraries loaded. Ready to begin.


---
## PART 3 — Reconciliation Workflow

We compare two data sources:
- **ledger.csv** — Internal payment records (what we recorded)
- **gateway.csv** — External gateway records (what the payment gateway processed)

Goal: Find differences, flag issues, produce a reconciliation report.

### Step 1 — Load Both Files

In [2]:
ledger  = pd.read_csv(os.path.join(RAW, 'ledger.csv'))
gateway = pd.read_csv(os.path.join(RAW, 'gateway.csv'))

print('=== LEDGER ===')
print(f'Rows: {len(ledger)} | Columns: {list(ledger.columns)}')
display(ledger)

print('\n=== GATEWAY ===')
print(f'Rows: {len(gateway)} | Columns: {list(gateway.columns)}')
display(gateway)

=== LEDGER ===
Rows: 10 | Columns: ['transaction_id', 'transaction_date', 'merchant_id', 'amount_usd', 'status', 'payment_method']


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,850.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R004,2026-03-02,M003,2100.0,success,Card
4,R005,2026-03-03,M004,7200.0,success,Card
5,R006,2026-03-03,M002,950.0,success,UPI
6,R007,2026-03-04,M005,3300.0,failed,NetBanking
7,R008,2026-03-04,M001,640.0,success,Card
8,R009,2026-03-05,M002,4100.0,success,Card
9,R010,2026-03-05,M004,2500.0,success,Wallet



=== GATEWAY ===
Rows: 9 | Columns: ['transaction_id', 'transaction_date', 'merchant_id', 'amount_usd', 'status', 'payment_method']


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,900.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R005,2026-03-03,M004,7200.0,failed,Card
4,R006,2026-03-03,M002,950.0,success,UPI
5,R007,2026-03-04,M005,3300.0,failed,NetBanking
6,R008,2026-03-04,M001,600.0,success,Card
7,R009,2026-03-05,M002,4100.0,success,Card
8,R011,2026-03-05,M003,1800.0,success,Card


### Step 2 — Check for Duplicates and Nulls

In [3]:
print('=== LEDGER ===')
print(f'Duplicate transaction IDs : {ledger.duplicated("transaction_id").sum()}')
print(f'Null values:\n{ledger.isnull().sum()}')

print('\n=== GATEWAY ===')
print(f'Duplicate transaction IDs : {gateway.duplicated("transaction_id").sum()}')
print(f'Null values:\n{gateway.isnull().sum()}')

=== LEDGER ===
Duplicate transaction IDs : 0
Null values:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

=== GATEWAY ===
Duplicate transaction IDs : 0
Null values:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64


### Step 3 — Identify Records Missing in Gateway
These are transactions that exist in our internal ledger but were **not found in the gateway** — possible settlement failures.

In [4]:
missing_in_gateway = ledger[~ledger['transaction_id'].isin(gateway['transaction_id'])].copy()
missing_in_gateway['issue'] = 'Present in Ledger but missing in Gateway'
print(f'Transactions missing in Gateway: {len(missing_in_gateway)}')
display(missing_in_gateway)

missing_in_gateway.to_csv(os.path.join(PROC, 'missing_in_gateway.csv'), index=False)
print('✔ Saved: missing_in_gateway.csv')

Transactions missing in Gateway: 2


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method,issue
3,R004,2026-03-02,M003,2100.0,success,Card,Present in Ledger but missing in Gateway
9,R010,2026-03-05,M004,2500.0,success,Wallet,Present in Ledger but missing in Gateway


✔ Saved: missing_in_gateway.csv


### Step 4 — Identify Records Missing in Ledger
These exist in the gateway but **never made it into our internal ledger** — possible unrecorded payments.

In [5]:
missing_in_ledger = gateway[~gateway['transaction_id'].isin(ledger['transaction_id'])].copy()
missing_in_ledger['issue'] = 'Present in Gateway but missing in Ledger'
print(f'Transactions missing in Ledger: {len(missing_in_ledger)}')
display(missing_in_ledger)

missing_in_ledger.to_csv(os.path.join(PROC, 'missing_in_ledger.csv'), index=False)
print('✔ Saved: missing_in_ledger.csv')

Transactions missing in Ledger: 1


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method,issue
8,R011,2026-03-05,M003,1800.0,success,Card,Present in Gateway but missing in Ledger


✔ Saved: missing_in_ledger.csv


### Step 5 — Identify Amount Mismatches
Same transaction ID exists in both, but the amounts differ.

In [6]:
common = pd.merge(ledger, gateway, on='transaction_id', suffixes=('_ledger', '_gateway'))
amount_mismatches = common[abs(common['amount_usd_ledger'] - common['amount_usd_gateway']) > 0.01].copy()
amount_mismatches['amount_difference'] = (amount_mismatches['amount_usd_ledger'] - amount_mismatches['amount_usd_gateway']).round(2)
amount_mismatches['issue'] = 'Amount mismatch between Ledger and Gateway'

out_cols = ['transaction_id','transaction_date_ledger','amount_usd_ledger','amount_usd_gateway','amount_difference','issue']
amt_out = amount_mismatches[out_cols].copy()
amt_out.columns = ['transaction_id','transaction_date','amount_usd_ledger','amount_usd_gateway','amount_difference','issue']

print(f'Amount mismatches found: {len(amt_out)}')
display(amt_out)

amt_out.to_csv(os.path.join(PROC, 'amount_mismatches.csv'), index=False)
print('✔ Saved: amount_mismatches.csv')

Amount mismatches found: 2


,transaction_id,transaction_date,amount_usd_ledger,amount_usd_gateway,amount_difference,issue
1,R002,2026-03-01,850.0,900.0,-50.0,Amount mismatch between Ledger and Gateway
6,R008,2026-03-04,640.0,600.0,40.0,Amount mismatch between Ledger and Gateway


✔ Saved: amount_mismatches.csv


### Step 6 — Identify Status Mismatches
Same transaction ID, different status in ledger vs gateway.

In [7]:
status_mismatches = common[common['status_ledger'] != common['status_gateway']].copy()
status_mismatches['issue'] = 'Status mismatch between Ledger and Gateway'

sts_out = status_mismatches[['transaction_id','transaction_date_ledger','status_ledger','status_gateway','issue']].copy()
sts_out.columns = ['transaction_id','transaction_date','status_ledger','status_gateway','issue']

print(f'Status mismatches found: {len(sts_out)}')
display(sts_out)

sts_out.to_csv(os.path.join(PROC, 'status_mismatches.csv'), index=False)
print('✔ Saved: status_mismatches.csv')

Status mismatches found: 1


,transaction_id,transaction_date,status_ledger,status_gateway,issue
3,R005,2026-03-03,success,failed,Status mismatch between Ledger and Gateway


✔ Saved: status_mismatches.csv


### Step 7 — Build Final Reconciliation Report

In [8]:
all_ids = sorted(set(ledger['transaction_id']) | set(gateway['transaction_id']))
recon_rows = []

for tid in all_ids:
    l = ledger[ledger['transaction_id'] == tid]
    g = gateway[gateway['transaction_id'] == tid]
    in_l, in_g = not l.empty, not g.empty

    if in_l and in_g:
        l_amt = float(l['amount_usd'].iloc[0])
        g_amt = float(g['amount_usd'].iloc[0])
        l_sts = l['status'].iloc[0]
        g_sts = g['status'].iloc[0]
        amt_ok = abs(l_amt - g_amt) < 0.01
        sts_ok = l_sts == g_sts
        if amt_ok and sts_ok:         rec = 'MATCHED'
        elif not amt_ok and not sts_ok: rec = 'AMOUNT & STATUS MISMATCH'
        elif not amt_ok:              rec = 'AMOUNT MISMATCH'
        else:                         rec = 'STATUS MISMATCH'
        recon_rows.append({'transaction_id': tid, 'in_ledger': 'YES', 'in_gateway': 'YES',
            'ledger_amount': l_amt, 'gateway_amount': g_amt, 'amount_diff': round(l_amt-g_amt,2),
            'ledger_status': l_sts, 'gateway_status': g_sts, 'reconciliation_status': rec})
    elif in_l:
        recon_rows.append({'transaction_id': tid, 'in_ledger': 'YES', 'in_gateway': 'NO',
            'ledger_amount': float(l['amount_usd'].iloc[0]), 'gateway_amount': None, 'amount_diff': None,
            'ledger_status': l['status'].iloc[0], 'gateway_status': None, 'reconciliation_status': 'MISSING IN GATEWAY'})
    else:
        recon_rows.append({'transaction_id': tid, 'in_ledger': 'NO', 'in_gateway': 'YES',
            'ledger_amount': None, 'gateway_amount': float(g['amount_usd'].iloc[0]), 'amount_diff': None,
            'ledger_status': None, 'gateway_status': g['status'].iloc[0], 'reconciliation_status': 'MISSING IN LEDGER'})

recon_df = pd.DataFrame(recon_rows)
recon_df.to_csv(os.path.join(PROC, 'reconciliation_report.csv'), index=False)

print('=== FINAL RECONCILIATION REPORT ===')
display(recon_df)
print('\nSummary:')
print(recon_df['reconciliation_status'].value_counts().to_string())
print('\n✔ Saved: reconciliation_report.csv')

=== FINAL RECONCILIATION REPORT ===


,transaction_id,in_ledger,in_gateway,ledger_amount,gateway_amount,amount_diff,ledger_status,gateway_status,reconciliation_status
0,R001,YES,YES,1200.0,1200.0,0.0,success,success,MATCHED
1,R002,YES,YES,850.0,900.0,-50.0,success,success,AMOUNT MISMATCH
2,R003,YES,YES,500.0,500.0,0.0,success,success,MATCHED
3,R004,YES,NO,2100.0,NaN,NaN,success,NaN,MISSING IN GATEWAY
4,R005,YES,YES,7200.0,7200.0,0.0,success,failed,STATUS MISMATCH
5,R006,YES,YES,950.0,950.0,0.0,success,success,MATCHED
6,R007,YES,YES,3300.0,3300.0,0.0,failed,failed,MATCHED
7,R008,YES,YES,640.0,600.0,40.0,success,success,AMOUNT MISMATCH
8,R009,YES,YES,4100.0,4100.0,0.0,success,success,MATCHED
9,R010,YES,NO,2500.0,NaN,NaN,success,NaN,MISSING IN GATEWAY



Summary:
reconciliation_status
MATCHED               5
AMOUNT MISMATCH       2
MISSING IN GATEWAY    2
STATUS MISMATCH       1
MISSING IN LEDGER     1

✔ Saved: reconciliation_report.csv


### Step 8 — Generate Summary Metrics JSON

In [9]:
summary_metrics = {
    "total_ledger_rows":          len(ledger),
    "total_gateway_rows":         len(gateway),
    "missing_in_gateway_count":   len(missing_in_gateway),
    "missing_in_ledger_count":    len(missing_in_ledger),
    "amount_mismatch_count":      len(amount_mismatches),
    "status_mismatch_count":      len(status_mismatches),
    "reconciliation_issue_count": len(recon_df[recon_df['reconciliation_status'] != 'MATCHED']),
    "ledger_total_amount":        round(float(ledger['amount_usd'].sum()), 2),
    "gateway_total_amount":       round(float(gateway['amount_usd'].sum()), 2),
    "amount_at_risk":             round(float(amt_out['amount_usd_ledger'].sum()) if len(amt_out) else 0.0, 2),
}

with open(os.path.join('..', '04_python', 'summary_metrics.json'), 'w') as f:
    json.dump(summary_metrics, f, indent=2)

print('Summary Metrics:')
print(json.dumps(summary_metrics, indent=2))
print('\n✔ Saved: 04_python/summary_metrics.json')

Summary Metrics:
{
  "total_ledger_rows": 10,
  "total_gateway_rows": 9,
  "missing_in_gateway_count": 2,
  "missing_in_ledger_count": 1,
  "amount_mismatch_count": 2,
  "status_mismatch_count": 1,
  "reconciliation_issue_count": 6,
  "ledger_total_amount": 23340.0,
  "gateway_total_amount": 20550.0,
  "amount_at_risk": 1490.0
}

✔ Saved: 04_python/summary_metrics.json


---
## PART 4 — JSON Normalization

The `api_response_sample.json` is a **nested JSON** (batches → settlements → bank details).  
We flatten it into a clean tabular CSV.

### Step 1 — Read the Nested JSON

In [10]:
with open(os.path.join(RAW, 'api_response_sample.json')) as f:
    api_data = json.load(f)

print('JSON Structure:')
print(f"  generated_at : {api_data['generated_at']}")
print(f"  source       : {api_data['source']}")
print(f"  total batches: {len(api_data['batches'])}")
for batch in api_data['batches']:
    print(f"  batch {batch['batch_id']} → merchant {batch['merchant']['merchant_name']} → {len(batch['settlements'])} settlements")

JSON Structure:
  generated_at : 2026-03-07T10:00:00Z
  source       : QuickPay Settlement API
  total batches: 2
  batch B001 → merchant Alpha Mart → 3 settlements
  batch B002 → merchant Delta Travels → 3 settlements


### Step 2 — Flatten into Tabular Form

In [11]:
rows = []
for batch in api_data['batches']:
    for sett in batch['settlements']:
        rows.append({
            'batch_id':        batch['batch_id'],
            'merchant_id':     batch['merchant']['merchant_id'],
            'merchant_name':   batch['merchant']['merchant_name'],
            'merchant_region': batch['merchant']['region'],
            'settlement_id':   sett['settlement_id'],
            'amount_usd':      sett['amount_usd'],
            'status':          sett['status'],
            'processed_at':    sett['processed_at'],
            'bank_name':       sett['bank']['name'],
            'bank_country':    sett['bank']['country'],
        })

api_df = pd.DataFrame(rows)
print(f'Flattened: {len(api_df)} rows from {len(api_data["batches"])} batches')
display(api_df)

Flattened: 6 rows from 2 batches


,batch_id,merchant_id,merchant_name,merchant_region,settlement_id,amount_usd,status,processed_at,bank_name,bank_country
0,B001,M001,Alpha Mart,APAC,S001,1520.5,settled,2026-03-07T08:10:00Z,Bank A,IN
1,B001,M001,Alpha Mart,APAC,S002,980.0,pending,2026-03-07T08:45:00Z,Bank A,IN
2,B001,M001,Alpha Mart,APAC,S003,640.0,settled,2026-03-07T09:15:00Z,Bank B,SG
3,B002,M004,Delta Travels,US,S004,2100.0,settled,2026-03-07T08:20:00Z,Bank C,US
4,B002,M004,Delta Travels,US,S005,500.0,failed,2026-03-07T08:50:00Z,Bank C,US
5,B002,M004,Delta Travels,US,S006,7200.0,settled,2026-03-07T09:30:00Z,Bank C,US


### Step 3 — Clean Column Names

In [12]:
api_df.columns = [c.lower().replace(' ', '_') for c in api_df.columns]
print('Column names (cleaned):', list(api_df.columns))

Column names (cleaned): ['batch_id', 'merchant_id', 'merchant_name', 'merchant_region', 'settlement_id', 'amount_usd', 'status', 'processed_at', 'bank_name', 'bank_country']


### Step 4 — Convert Date/Time Fields

In [13]:
api_df['processed_at']   = pd.to_datetime(api_df['processed_at'])
api_df['processed_date'] = api_df['processed_at'].dt.strftime('%Y-%m-%d')
api_df['processed_time'] = api_df['processed_at'].dt.strftime('%H:%M:%S')
print('Date/time columns added')
display(api_df[['settlement_id','processed_at','processed_date','processed_time']])

Date/time columns added

,settlement_id,processed_at,processed_date,processed_time
0,S001,2026-03-07 08:10:00+00:00,2026-03-07,08:10:00
1,S002,2026-03-07 08:45:00+00:00,2026-03-07,08:45:00
2,S003,2026-03-07 09:15:00+00:00,2026-03-07,09:15:00
3,S004,2026-03-07 08:20:00+00:00,2026-03-07,08:20:00
4,S005,2026-03-07 08:50:00+00:00,2026-03-07,08:50:00
5,S006,2026-03-07 09:30:00+00:00,2026-03-07,09:30:00


### Step 5 — Save Normalized Output

In [14]:
api_df.to_csv(os.path.join(PROC, 'api_normalized.csv'), index=False)
print(f'✔ Saved: api_normalized.csv ({len(api_df)} rows)')
print('\nFinal normalized table:')
display(api_df)
print('\nColumn dtypes:')
print(api_df.dtypes.to_string())

✔ Saved: api_normalized.csv (6 rows)

Final normalized table:


,batch_id,merchant_id,merchant_name,merchant_region,settlement_id,amount_usd,status,processed_at,bank_name,bank_country,processed_date,processed_time
0,B001,M001,Alpha Mart,APAC,S001,1520.5,settled,2026-03-07 08:10:00+00:00,Bank A,IN,2026-03-07,08:10:00
1,B001,M001,Alpha Mart,APAC,S002,980.0,pending,2026-03-07 08:45:00+00:00,Bank A,IN,2026-03-07,08:45:00
2,B001,M001,Alpha Mart,APAC,S003,640.0,settled,2026-03-07 09:15:00+00:00,Bank B,SG,2026-03-07,09:15:00
3,B002,M004,Delta Travels,US,S004,2100.0,settled,2026-03-07 08:20:00+00:00,Bank C,US,2026-03-07,08:20:00
4,B002,M004,Delta Travels,US,S005,500.0,failed,2026-03-07 08:50:00+00:00,Bank C,US,2026-03-07,08:50:00
5,B002,M004,Delta Travels,US,S006,7200.0,settled,2026-03-07 09:30:00+00:00,Bank C,US,2026-03-07,09:30:00



Column dtypes:
batch_id                           str
merchant_id                        str
merchant_name                      str
merchant_region                    str
settlement_id                      str
amount_usd                     float64
status                             str
processed_at       datetime64[us, UTC]
bank_name                          str
bank_country                       str
processed_date                     str
processed_time                     str


---
## Supporting Data for Dashboard (Part 5)

The following CSVs were generated in `run_all.py` and are available in `01_data/processed/`:

| File | Purpose |
|---|---|
| `cleaned_transactions.csv` | Full transaction data with all flags |
| `daily_summary.csv` | GMV and counts per day (trend chart) |
| `payment_method_breakdown.csv` | Breakdown by UPI, Card, Wallet etc. |
| `region_breakdown.csv` | APAC / EU / US performance |
| `merchant_performance_summary.csv` | Per-merchant GMV and risk |
| `kpi_summary.csv` | Headline KPI numbers |